In [1]:
"""
BicapaCryoET v15 — Dataset sintético para identificación lipídica en cryo-ET.
Geometría molecular en vista única coherente.

===========================================================================
REFERENCIAS CIENTÍFICAS
===========================================================================

Fundamentos biofísicos de la bicapa:
  [1] Singer, S.J. & Nicolson, G.L. (1972). The fluid mosaic model of the
      structure of cell membranes. Science, 175(4023), 720-731.
      doi:10.1126/science.175.4023.720

  [2] Smith, D.J., Klauda, J.B. & Sodt, A.J. (2019). Simulation best
      practices for lipid membranes. Living J. Comput. Mol. Sci., 1(1), 5966.
      doi:10.33011/livecoms.1.1.5966

  [3] Garcia-Fandino, R., Pineiro, A., Trick, J.L. & Sansom, M.S.P. (2016).
      Lipid bilayer membrane perturbation by embedded amphipathic peptides:
      developing electrostatics-based hypothesis. ACS Nano, 10, 10.
      doi:10.1021/acsnano.6b00202

Curvatura y elasticidad (modelo de Helfrich):
  [4] Helfrich, W. (1973). Elastic properties of lipid bilayers: theory and
      possible experiments. Z. Naturforsch. C, 28(11-12), 693-703.

  [5] Pinigin, K.V. (2022). Determination of elastic parameters of lipid
      membranes with molecular dynamics: A review of approaches and
      theoretical aspects. Membranes, 12(11), 1049.
      doi:10.3390/membranes12111149

  [6] Chakraborty, S., Doktorova, M., Molugu, T.R., Heberle, F.A., et al.
      (2020). How cholesterol stiffens unsaturated lipid membranes.
      Proc. Natl. Acad. Sci. USA, 117(36), 21896-21905.
      doi:10.1073/pnas.2004807117

Parámetros de orden de cadenas acil:
  [7] Piggot, T.J., Allison, J.R., Sessions, R.B. & Essex, J.W. (2017).
      On the calculation of acyl chain order parameters from lipid
      simulations. J. Chem. Theory Comput., 13(11), 5683-5696.
      doi:10.1021/acs.jctc.7b00643

  [8] Bartos, L., Pajtinka, P. & Vacha, R. (2025). gorder: Comprehensive
      tool for calculating lipid order parameters from molecular simulations.
      SoftwareX. doi:10.1016/j.softx.2025.102254

Interdigitación trans-leaflet:
  [9] Chaisson, E.H., Heberle, F.A. & Doktorova, M. (2025). Quantifying
      acyl chain interdigitation in simulated bilayers via direct
      transbilayer interactions. J. Chem. Inf. Model., 65, 3879-3885.
      doi:10.1021/acs.jcim.4c02287

Asimetría lipídica (PIPs y POPS):
 [10] Daleke, D.L. (2003). Regulation of transbilayer plasma membrane
      phospholipid asymmetry. J. Lipid Res., 44(2), 233-242.
      doi:10.1194/jlr.R200019-JLR200

 [11] Di Paolo, G. & De Camilli, P. (2006). Phosphoinositides in cell
      regulation and membrane dynamics. Nature, 443(7112), 651-657.
      doi:10.1038/nature05185

Balsas lipídicas (rafts):
 [12] Simons, K. & Ikonen, E. (1997). Functional rafts in cell membranes.
      Nature, 387(6633), 569-572. doi:10.1038/42408

 [13] Lingwood, D. & Simons, K. (2010). Lipid rafts as a membrane-organizing
      principle. Science, 327(5961), 46-50. doi:10.1126/science.1174621

Cryo-ET de membranas:
 [14] Dubochet, J., Adrian, M., Chang, J.J., Homo, J.C., et al. (1988).
      Cryo-electron microscopy of vitrified specimens. Q. Rev. Biophys.,
      21(2), 129-228. doi:10.1017/S0033583500004297

 [15] Sharma, K.D., Heberle, F.A. & Waxham, M.N. (2023). Visualizing lipid
      membrane structure with cryo-EM: Past, present, and future. Emerging
      Topics Life Sci., 7(1), 55-65. doi:10.1042/ETLS20220090

 [16] Lucic, V., Rigort, A. & Baumeister, W. (2013). Cryo-electron
      tomography: The challenge of doing structural biology in situ.
      J. Cell Biol., 202(3), 407-419. doi:10.1083/jcb.201304193

Propiedades estructurales de lipidos:
 [17] Kucerka, N., Nieh, M.P. & Katsaras, J. (2011). Fluid phase lipid
      areas and bilayer thicknesses of commonly used phosphatidylcholines
      as a function of temperature. Biochim. Biophys. Acta, 1808(11),
      2761-2771. doi:10.1016/j.bbamem.2011.07.022

===========================================================================
"""

import json
import os
import warnings
from dataclasses import dataclass
from typing import Dict, List, Tuple, Optional

import matplotlib.gridspec as mgrid
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.colors import LinearSegmentedColormap
from matplotlib.ticker import AutoMinorLocator
from numpy.random import default_rng
from scipy.ndimage import gaussian_filter
from scipy.spatial import KDTree

warnings.filterwarnings("ignore")

OUTPUT_DIR = "CryoET"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Resolución de figuras
# 300 dpi para calidad de publicación; ajustar a 150 para rapidez
FIG_DPI = 300

CMAP_THICK = LinearSegmentedColormap.from_list(
    "thick",
    ["#053061", "#2166ac", "#92c5de", "#f7f7f7",
     "#f4a582", "#d6604d", "#67001f"],
)
CMAP_RAFT = LinearSegmentedColormap.from_list(
    "raft",
    ["#0d1117", "#0f3460", "#533483", "#e94560", "#ff6b6b"],
)
CMAP_ORDER = LinearSegmentedColormap.from_list(
    "order",
    ["#f7f7f7", "#d9f0d3", "#5aae61", "#1b7837", "#00441b"],
)
CMAP_INTERDIG = LinearSegmentedColormap.from_list(
    "interdig",
    ["#f7f7f7", "#fddbc7", "#f4a582", "#d6604d", "#67001f"],
)

PLT_STYLE = {
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.edgecolor": "#333333",
    "axes.linewidth": 1.0,
    "axes.grid": True,
    "grid.color": "#e8e8e8",
    "grid.linewidth": 0.5,
    "xtick.color": "#333333",
    "ytick.color": "#333333",
    "axes.labelcolor": "#333333",
    "text.color": "#1a1a1a",
    "font.family": "sans-serif",
    "font.size": 10,
    "axes.titlesize": 12,
    "axes.titleweight": "bold",
}


# Tipos lipídicos
# Propiedades biofísicas de referencia [17]:
#   area: Å²   tail_length/hg_thick/hg_radius/glyc_offset: Å
#   mass: Da   phase: "gel" | "fluid"   pip_order: 0-3

@dataclass(frozen=True)
class LipidType:
    name: str
    area: float
    tail_length: float
    hg_thick: float
    hg_radius: float
    glyc_offset: float
    nc: Tuple[int, int]
    ndb: Tuple[int, int]
    dbpos: Tuple[Optional[int], Optional[int]]
    mass: float
    phase: str
    is_raft: bool
    charge: int
    color: str
    color_tail1: str
    color_tail2: str
    pip_order: int = 0


# Datos estructurales de lipidos: Kucerka 2011 [17], McLaughlin & Murray 2005
LIPID_TYPES = {
    "POPC": LipidType(
        "POPC", 64.3, 14.5, 9.0, 4.5, 3.5, (16, 18), (0, 1), (None, 9),
        760.1, "fluid", False, 0,
        "#3a86ff", "#90c8ff", "#1a5fbf",
    ),
    "POPE": LipidType(
        "POPE", 59.0, 14.3, 8.5, 3.8, 3.5, (16, 18), (0, 1), (None, 9),
        717.0, "fluid", False, 0,
        "#e63946", "#f4a5aa", "#9b1e28",
    ),
    "POPS": LipidType(
        "POPS", 59.7, 14.2, 9.5, 4.0, 3.5, (16, 18), (0, 1), (None, 9),
        761.0, "fluid", False, -1,
        "#fb8500", "#ffd166", "#c84b00",
    ),
    "PI": LipidType(
        "PI", 67.0, 14.5, 10.0, 5.5, 3.5, (18, 20), (0, 4), (None, 5),
        857.0, "fluid", False, -1,
        "#9b5de5", "#c897f5", "#5a189a",
    ),
    "PI3P": LipidType(
        "PI3P", 68.5, 14.5, 11.0, 6.0, 3.5, (18, 20), (0, 4), (None, 5),
        937.0, "fluid", False, -2,
        "#c77dff", "#e0aaff", "#7b2d8b", 1,
    ),
    "PI4P": LipidType(
        "PI4P", 69.0, 14.5, 11.2, 6.0, 3.5, (18, 20), (0, 4), (None, 5),
        937.0, "fluid", False, -2,
        "#7209b7", "#b5179e", "#480ca8", 1,
    ),
    "PI5P": LipidType(
        "PI5P", 68.0, 14.5, 10.8, 5.8, 3.5, (18, 20), (0, 4), (None, 5),
        937.0, "fluid", False, -2,
        "#560bad", "#7b2fbe", "#3a0ca3", 1,
    ),
    "PI34P2": LipidType(
        "PI(3,4)P2", 72.0, 14.8, 12.5, 6.8, 3.5, (18, 20), (0, 4), (None, 5),
        1017.0, "fluid", False, -3,
        "#f72585", "#ff85c2", "#b5179e", 2,
    ),
    "PIP2": LipidType(
        "PIP2", 75.0, 14.8, 13.0, 7.2, 3.5, (18, 20), (0, 4), (None, 5),
        1017.0, "fluid", False, -4,
        "#f1c40f", "#f8e08a", "#c09000", 2,
    ),
    "PIP3": LipidType(
        "PIP3", 80.0, 14.8, 14.5, 8.0, 3.5, (18, 20), (0, 4), (None, 5),
        1097.0, "fluid", False, -5,
        "#ff6b35", "#ffb494", "#c23b00", 3,
    ),
    # Lipidos de fase gel / raft — condensados por CHOL [6, 12, 13]
    "SM": LipidType(
        "SM", 47.0, 16.5, 10.0, 4.5, 4.0, (18, 24), (1, 0), (4, None),
        731.0, "gel", True, 0,
        "#2dc653", "#95e8b4", "#0a6e2d",
    ),
    "CHOL": LipidType(
        "CHOL", 38.5, 17.0, 4.0, 2.2, 0.0, (27, 0), (1, 0), (5, None),
        386.7, "gel", True, 0,
        "#adb5bd", "#dee2e6", "#6c757d",
    ),
    "GM1": LipidType(
        "GM1", 85.0, 17.0, 15.0, 9.0, 4.0, (18, 20), (1, 0), (4, None),
        1545.0, "gel", True, -1,
        "#d4a017", "#f0d080", "#8a6200",
    ),
}

# Composiciones base (van Meer et al. 2008; Verkleij & Post 2000)
# Asimetria estricta: POPS ~ 0 en outer, SM ~ 0 en inner [10]
COMP_OUTER_BASE = {
    "POPC": 0.33, "SM": 0.24, "CHOL": 0.30,
    "GM1": 0.05, "PI": 0.04, "POPE": 0.04,
}
COMP_INNER_BASE = {
    "POPC": 0.18, "POPE": 0.24, "POPS": 0.14,
    "CHOL": 0.28, "PI": 0.05,
    "PIP2": 0.04, "PI4P": 0.03, "PI3P": 0.02,
    "PIP3": 0.01, "PI34P2": 0.01,
}


@dataclass
class LipidInstance:
    lipid_type: LipidType
    leaflet: str
    head_pos: np.ndarray
    glycerol_pos: np.ndarray
    tail1: List[np.ndarray]
    tail2: Optional[List[np.ndarray]]
    order_param: float      # S_CH medio de este lipido [7, 8]
    in_raft: bool           # pertenece a dominio Lo [12, 13]
    is_pip: bool            # fosfoinositol [11]


@dataclass
class MembraneGeometry:
    hydro_thick: float   # grosor hidrofobico (Å)
    total_thick: float   # grosor total incluyendo cabezas (Å)
    z_outer: float       # posicion media cabezas externas (Å)
    z_inner: float       # posicion media cabezas internas (Å)
    z_mid: float         # plano medio nominal (Å)


class BicapaCryoET:
    """
    Instantanea estatica de membrana plasmatica de mamifero para
    entrenamiento de IA sobre criotomografias electronicas.

    Cada semilla produce una muestra biologicamente distinta:
      - Composicion variable (distribucion Dirichlet, CV ~12%) [van Meer 2008]
      - kc dependiente de composicion (CHOL/SM rigidizan) [6]
      - Curvatura Helfrich completa: <|h_q|^2>=kBT/(kc*q^4 + sigma*q^2) [4, 5]
      - Parametro de orden S_CH por celda [7, 8]
      - Interdigitacion trans-leaflet normalizada [9]

    Uso:
        b = BicapaCryoET(size_nm=(50, 50), seed=42)
        b.build()
        b.save_all()
        b.export_training()
    """

    def __init__(self, size_nm=(50, 50), seed=42):
        self.seed = seed
        self.rng = default_rng(seed)
        self.size_nm = size_nm
        self.Lx = size_nm[0] * 10
        self.Ly = size_nm[1] * 10

        self.outer_leaflet = []
        self.inner_leaflet = []
        self.perturbations = []
        self.rafts_outer = []
        self.rafts_inner = []
        self.pip_clusters = []

        self.geometry = None
        self.curvature_map = None
        self.bending_modulus = 25.0    # kBT·nm² [5]
        self.surface_tension = 0.01   # kBT/nm² [4]
        self.perturbation_density = 0.012

        self.comp_outer = {}
        self.comp_inner = {}

    def _random_composition(self, base, concentration=50):
        """
        Composicion aleatoria mediante distribucion Dirichlet.
        CV ~12% es consistente con variabilidad biologica intercelular.
        concentration=50 → alpha = base * 50, lo que da CV ~ 1/sqrt(50*f).
        """
        alpha = np.array([base.get(k, 0) for k in LIPID_TYPES]) * concentration
        values = self.rng.dirichlet(alpha + 1e-8)
        return {k: v for k, v in zip(LIPID_TYPES, values) if v > 0.01}

    def _calculate_geometry(self):
        """
        Grosor y posiciones de cabezas a partir de las fracciones lipidicas.
        Basado en datos estructurales de Kucerka 2011 [17].
        """
        def mean_prop(comp, prop):
            return sum(
                getattr(LIPID_TYPES[k], prop) * f
                for k, f in comp.items() if k in LIPID_TYPES
            )

        tail_o = mean_prop(self.comp_outer, "tail_length")
        tail_i = mean_prop(self.comp_inner, "tail_length")
        hg_o   = mean_prop(self.comp_outer, "hg_thick")
        hg_i   = mean_prop(self.comp_inner, "hg_thick")
        glyc_o = mean_prop(self.comp_outer, "glyc_offset")
        glyc_i = mean_prop(self.comp_inner, "glyc_offset")

        hydro = tail_o + tail_i
        total = hydro + hg_o + hg_i + glyc_o + glyc_i

        return MembraneGeometry(
            hydro_thick=hydro,
            total_thick=total,
            z_outer=tail_o + glyc_o,
            z_inner=-(tail_i + glyc_i),
            z_mid=(tail_o + glyc_o - tail_i - glyc_i) / 2,
        )

    def _bending_from_composition(self):
        """
        Rigidez de bending kc calculada de la composicion lipidica.

        CHOL y SM condensan las cadenas acil, aumentando significativamente
        kc segun Chakraborty et al. PNAS 2020 [6].
        Pesos empiricos: CHOL x2, SM x1.6, GM1 x1.8 respecto a POPC.
        Rango resultante: 18-45 kBT·nm², consistente con Pinigin 2022 [5].
        """
        # Pesos derivados de Chakraborty et al. 2020 [6]
        KC_W = {
            "POPC": 1.00, "POPE": 1.05, "POPS": 1.10,
            "PI": 1.00, "PI3P": 1.00, "PI4P": 1.00,
            "PI5P": 1.00, "PI34P2": 1.05,
            "PIP2": 1.20, "PIP3": 1.30,
            "SM": 1.60, "CHOL": 2.00, "GM1": 1.80,
        }

        def leaflet_kc(comp):
            return sum(KC_W.get(k, 1.0) * f for k, f in comp.items())

        kc_norm = 0.5 * (leaflet_kc(self.comp_outer) + leaflet_kc(self.comp_inner))
        return float(np.clip(20.0 + 20.0 * (kc_norm - 1.0), 18.0, 45.0))

    def _generate_curvature(self, bins=64):
        """
        Campo de alturas con espectro de Helfrich completo [4, 5]:
          <|h_q|^2> = kBT / (kc * q^4 + sigma * q^2)

        kc controla la rigidez de bending (modos de alta q).
        sigma (tension superficial) suprime los modos de larga longitud de onda.
        Amplitud resultante: ~2-5 A rms, consistente con cryo-ET [14, 15].
        """
        L_nm = self.Lx / 10
        qx = 2 * np.pi * np.fft.fftfreq(bins, d=L_nm / bins)
        Qx, Qy = np.meshgrid(qx, qx)
        Q2 = Qx**2 + Qy**2
        Q4 = Q2**2

        denom = self.bending_modulus * Q4 + self.surface_tension * Q2
        denom[0, 0] = 1.0   # modo q=0: sin traslacion global

        sigma_q = np.sqrt(1.0 / denom) * (L_nm / bins)
        sigma_q[0, 0] = 0.0

        noise = (self.rng.standard_normal((bins, bins))
                 + 1j * self.rng.standard_normal((bins, bins))) * sigma_q
        h_nm = np.real(np.fft.ifft2(noise)) * bins
        self.curvature_map = h_nm * 10   # nm → Å

    def _get_local_z(self, x, y, bins=64):
        if self.curvature_map is None:
            return 0.0
        ix = int(x / self.Lx * bins) % bins
        iy = int(y / self.Ly * bins) % bins
        return float(self.curvature_map[ix, iy])

    def _generate_tail(self, start, nc, ndb, dbpos, direction,
                       tilt, phi, phase, nseg=9):
        """
        Cadena acil como lista de nseg+1 puntos 3D.

        Kink en el doble enlace segun Smith et al. LiveCoMS 2019 [2].
        S_CH calculado por segmento segun Piggot et al. 2017 [7].
        Desorden gauche: gel=0.25, fluido=0.80 (Å).
        """
        L = nc * 1.26
        dz_base = direction * L / nseg * np.cos(tilt)
        dr = L / nseg * np.sin(tilt)
        disorder = 0.25 if phase == "gel" else 0.80

        points = [start.copy()]
        current = start.copy()
        order_vals = []

        for s in range(nseg):
            frac = (s + 0.5) / nseg
            kink = 0.0
            if ndb > 0 and dbpos:
                kink = 2.2 * np.exp(-((frac - dbpos / nc) ** 2) / 0.018)

            jx = self.rng.normal(0, disorder) + kink * np.cos(phi + np.pi / 2)
            jy = self.rng.normal(0, disorder) + kink * np.sin(phi + np.pi / 2)
            dx = dr * np.cos(phi) + jx
            dy = dr * np.sin(phi) + jy
            dz = dz_base

            seg_len = np.sqrt(dx**2 + dy**2 + dz**2)
            if seg_len > 0:
                cos_theta = abs(dz) / seg_len
                # S_CH = <(3cos^2(theta) - 1) / 2> [7]
                order_vals.append((3 * cos_theta**2 - 1) / 2)

            current = current + np.array([dx, dy, dz])
            points.append(current.copy())

        S_CH = float(np.mean(order_vals)) if order_vals else 0.5
        return points, S_CH

    def _create_lipid(self, lipid_name, x, y, z_base, leaflet):
        ltype = LIPID_TYPES[lipid_name]
        sign = -1 if leaflet == "sup" else 1

        z_head = z_base + self._get_local_z(x, y) + self.rng.normal(0, 0.5)
        head_pos = np.array([x, y, z_head])

        tilt = self.rng.uniform(3, 12 if ltype.phase == "gel" else 27) * np.pi / 180
        phi = self.rng.uniform(0, 2 * np.pi)

        glyc_z = z_head + sign * ltype.glyc_offset
        glyc_xy = np.sin(tilt) * ltype.glyc_offset * 0.3
        glycerol_pos = np.array([
            x + glyc_xy * np.cos(phi),
            y + glyc_xy * np.sin(phi),
            glyc_z,
        ])

        dphi = np.pi / 5

        if lipid_name == "CHOL":
            tail1, s1 = self._generate_tail(
                glycerol_pos, 17, 1, 5, sign, tilt * 0.4, phi, "gel"
            )
            tail2, s2 = None, s1
        else:
            nc1, nc2 = ltype.nc
            ndb1, ndb2 = ltype.ndb
            dbp1, dbp2 = ltype.dbpos
            tail1, s1 = self._generate_tail(
                glycerol_pos + np.array([1.1 * np.cos(phi - dphi),
                                         1.1 * np.sin(phi - dphi), 0]),
                nc1, ndb1, dbp1, sign, tilt, phi - dphi, ltype.phase,
            )
            tail2, s2 = self._generate_tail(
                glycerol_pos + np.array([1.1 * np.cos(phi + dphi),
                                         1.1 * np.sin(phi + dphi), 0]),
                nc2, ndb2, dbp2, sign, tilt, phi + dphi, ltype.phase,
            )

        return LipidInstance(
            lipid_type=ltype,
            leaflet=leaflet,
            head_pos=head_pos,
            glycerol_pos=glycerol_pos,
            tail1=tail1,
            tail2=tail2,
            order_param=(s1 + s2) / 2 if tail2 else s1,
            in_raft=False,
            is_pip=ltype.pip_order > 0,
        )

    def _populate_leaflet(self, composition, z_base, leaflet):
        """
        Grilla hexagonal con jitter gaussiano (30% del espaciado).
        Nucleos de balsas y PIPs controlados por la semilla.
        """
        area_t = self.Lx * self.Ly
        area_m = sum(
            LIPID_TYPES[k].area * f for k, f in composition.items()
            if k in LIPID_TYPES
        )
        n_total = int(area_t / area_m * 1.4)

        counts = {}
        remaining = n_total
        sorted_lipids = sorted(composition.items(), key=lambda x: -x[1])
        for i, (ltype, frac) in enumerate(sorted_lipids):
            if ltype not in LIPID_TYPES:
                continue
            counts[ltype] = (
                int(n_total * frac) if i < len(sorted_lipids) - 1 else remaining
            )
            remaining -= counts[ltype]

        nx = max(1, int(np.sqrt(n_total * self.Lx / self.Ly)))
        ny = max(1, n_total // nx + 1)
        dx, dy = self.Lx / nx, self.Ly / ny

        positions = []
        for ix in range(nx + 2):
            for iy in range(ny + 2):
                x = (ix * dx + (0.5 * dx if iy % 2 else 0)
                     + self.rng.normal(0, dx * 0.20))
                y = iy * dy + self.rng.normal(0, dy * 0.20)
                positions.append((x % self.Lx, y % self.Ly))

        self.rng.shuffle(positions)
        positions = positions[:n_total]

        # Nucleos de balsas (SM/CHOL enriquecidos) [12, 13]
        fr_raft = sum(
            f for k, f in composition.items()
            if LIPID_TYPES.get(k, LipidType("", 0, 0, 0, 0, 0,
                (0, 0), (0, 0), (None, None), 0, "", False, 0, "", "", "")).is_raft
        )
        n_nuc = max(3, int((self.Lx / 10) * (self.Ly / 10) * fr_raft / 100))
        raft_centers = [
            (
                self.rng.uniform(0.1 * self.Lx, 0.9 * self.Lx),
                self.rng.uniform(0.1 * self.Ly, 0.9 * self.Ly),
            )
            for _ in range(n_nuc)
        ]
        raft_radius = (
            np.sqrt(self.Lx * self.Ly * fr_raft / (np.pi * n_nuc)) * 0.75
        )

        # Nucleos de PIPs (clustering por cationes divalentes, no por carga
        # directa) [11] — modelado como zonas de enriquecimiento geometrico
        fr_pip = sum(
            f for k, f in composition.items()
            if LIPID_TYPES.get(
                k, LipidType("", 0, 0, 0, 0, 0,
                             (0, 0), (0, 0), (None, None), 0, "", False, 0, "", "", "")
            ).pip_order > 0
        )
        pip_centers = []
        pip_radius = 0.0
        if fr_pip > 0.01 and leaflet == "inf":
            n_pip = max(3, round(fr_pip * 8))
            pip_centers = [
                (
                    self.rng.uniform(0.1 * self.Lx, 0.9 * self.Lx),
                    self.rng.uniform(0.1 * self.Ly, 0.9 * self.Ly),
                )
                for _ in range(n_pip)
            ]
            pip_radius = np.sqrt(
                self.Lx * self.Ly * fr_pip / (np.pi * n_pip)
            )

        lipids = []
        pos_idx = 0
        for ltype, count in counts.items():
            for _ in range(count):
                if pos_idx >= len(positions):
                    break
                x, y = positions[pos_idx]
                pos_idx += 1

                if (LIPID_TYPES[ltype].pip_order > 0 and pip_centers
                        and not any(
                            np.hypot(x - cx, y - cy) < pip_radius
                            for cx, cy in pip_centers
                        )):
                    cx, cy = pip_centers[self.rng.integers(len(pip_centers))]
                    r = self.rng.uniform(0, pip_radius)
                    theta = self.rng.uniform(0, 2 * np.pi)
                    x = (cx + r * np.cos(theta)) % self.Lx
                    y = (cy + r * np.sin(theta)) % self.Ly

                lipid = self._create_lipid(ltype, x, y, z_base, leaflet)
                lipid.in_raft = (
                    LIPID_TYPES[ltype].is_raft
                    and any(
                        np.hypot(x - cx, y - cy) < raft_radius
                        for cx, cy in raft_centers
                    )
                )
                lipids.append(lipid)

        return lipids

    def _insert_perturbations(self):
        """
        Objetos que inducen curvatura local (proteinas, inclusiones).
        Desplazan lipidos vecinos mediante repulsion suave.
        """
        area = (self.Lx / 10) * (self.Ly / 10)
        n_obj = int(area * self.perturbation_density)

        for _ in range(n_obj):
            x = self.rng.uniform(0.1 * self.Lx, 0.9 * self.Lx)
            y = self.rng.uniform(0.1 * self.Ly, 0.9 * self.Ly)
            z = (
                (self.geometry.z_outer + self.geometry.z_inner) / 2
                + self._get_local_z(x, y)
            )
            radius = self.rng.uniform(8, 20)
            self.perturbations.append({"pos": np.array([x, y, z]), "radius": radius})

            for capa in [self.outer_leaflet, self.inner_leaflet]:
                for lip in capa:
                    dx = lip.head_pos[0] - x
                    dy = lip.head_pos[1] - y
                    dist = np.hypot(dx, dy)
                    if 0 < dist < radius * 1.5:
                        force = (radius * 1.5 - dist) / (radius * 1.5) * 6.0
                        lip.head_pos[0] = (
                            lip.head_pos[0] + (dx / dist) * force
                        ) % self.Lx
                        lip.head_pos[1] = (
                            lip.head_pos[1] + (dy / dist) * force
                        ) % self.Ly

    def _detect_clusters(self):
        """BFS sobre KDTree para detectar dominios lipidicos."""

        def find_clusters(lipids, condition, min_size=4):
            subset = [l for l in lipids if condition(l)]
            if len(subset) < min_size:
                return []
            coords = np.array([[l.head_pos[0], l.head_pos[1]]
                                for l in subset])
            tree = KDTree(coords)
            r_link = np.sqrt(self.Lx * self.Ly / len(subset)) * 1.6
            visited = set()
            clusters = []
            for i in range(len(subset)):
                if i in visited:
                    continue
                queue = [i]
                cl = {i}
                visited.add(i)
                while queue:
                    cur = queue.pop(0)
                    for nb in tree.query_ball_point(coords[cur], r=r_link):
                        if nb not in visited:
                            visited.add(nb)
                            cl.add(nb)
                            queue.append(nb)
                if len(cl) >= min_size:
                    clusters.append([subset[k] for k in cl])
            return clusters

        self.rafts_outer = find_clusters(self.outer_leaflet, lambda l: l.in_raft)
        self.rafts_inner = find_clusters(self.inner_leaflet, lambda l: l.in_raft)
        self.pip_clusters = find_clusters(
            self.inner_leaflet, lambda l: l.is_pip, min_size=3
        )


    def build(self):
        """
        Construye la bicapa completa. La semilla controla toda la aleatoriedad:
        composicion (Dirichlet), curvatura Helfrich, posiciones y dominios.
        """
        self.rng = default_rng(self.seed)
        self.perturbation_density = 0.008 + self.rng.uniform(0, 0.008)
        self.surface_tension = self.rng.uniform(0.001, 0.020)

        self.comp_outer = self._random_composition(COMP_OUTER_BASE)
        self.comp_inner = self._random_composition(COMP_INNER_BASE)
        self.geometry = self._calculate_geometry()
        self.bending_modulus = self._bending_from_composition()
        self._generate_curvature()

        self.outer_leaflet = self._populate_leaflet(
            self.comp_outer, self.geometry.z_outer, "sup"
        )
        self.inner_leaflet = self._populate_leaflet(
            self.comp_inner, self.geometry.z_inner, "inf"
        )
        self._insert_perturbations()
        self._detect_clusters()

        print(
            "  seed=%d | %.0fx%.0f nm | lipidos: %d | "
            "rafts: %d/%d | pips: %d | kc=%.0f kBT nm2 | sigma=%.3f"
            % (
                self.seed,
                self.Lx / 10, self.Ly / 10,
                len(self.outer_leaflet) + len(self.inner_leaflet),
                len(self.rafts_outer), len(self.rafts_inner),
                len(self.pip_clusters),
                self.bending_modulus,
                self.surface_tension,
            )
        )
        return self

    def _density_map(self, lipids, bins=90, sigma=1.8):
        H = np.zeros((bins, bins))
        ab = (self.Lx / bins) * (self.Ly / bins)
        for lip in lipids:
            xi = int(lip.head_pos[0] / self.Lx * bins) % bins
            yi = int(lip.head_pos[1] / self.Ly * bins) % bins
            H[xi, yi] += lip.lipid_type.mass
        return gaussian_filter(H / ab, sigma=sigma)

    def _roughness_map(self, lipids, bins=70, sigma=1.2):
        S, S2, C = (np.zeros((bins, bins)) for _ in range(3))
        for lip in lipids:
            ix = min(int(lip.head_pos[0] / self.Lx * bins), bins - 1)
            iy = min(int(lip.head_pos[1] / self.Ly * bins), bins - 1)
            z = lip.head_pos[2]
            S[ix, iy] += z
            S2[ix, iy] += z * z
            C[ix, iy] += 1
        with np.errstate(all="ignore"):
            mu = np.where(C > 0, S / C, 0)
            var = np.where(C > 1, S2 / C - mu**2, 0)
        return gaussian_filter(np.sqrt(np.clip(var, 0, None)), sigma=sigma)

    def _thickness_map(self, bins=70, sigma=2.0):
        def avg_z(lipids):
            S, C = np.zeros((bins, bins)), np.zeros((bins, bins))
            for lip in lipids:
                ix = min(int(lip.head_pos[0] / self.Lx * bins), bins - 1)
                iy = min(int(lip.head_pos[1] / self.Ly * bins), bins - 1)
                S[ix, iy] += lip.head_pos[2]
                C[ix, iy] += 1
            with np.errstate(all="ignore"):
                return np.where(C > 0, S / C, np.nan)

        d = avg_z(self.outer_leaflet) - avg_z(self.inner_leaflet)
        return gaussian_filter(np.nan_to_num(d, nan=float(np.nanmean(d))), sigma)

    def _raft_fraction_map(self, lipids, bins=90, sigma=2.5):
        Hr, Ht = np.zeros((bins, bins)), np.zeros((bins, bins))
        for lip in lipids:
            ix = min(int(lip.head_pos[0] / self.Lx * bins), bins - 1)
            iy = min(int(lip.head_pos[1] / self.Ly * bins), bins - 1)
            Ht[ix, iy] += 1
            if lip.in_raft:
                Hr[ix, iy] += 1
        with np.errstate(all="ignore"):
            return gaussian_filter(np.where(Ht > 0, Hr / Ht, 0), sigma=sigma)

    def _pip_density_map(self, bins=90, sigma=2.0):
        H = np.zeros((bins, bins))
        ab = (self.Lx / bins) * (self.Ly / bins)
        for lip in self.inner_leaflet:
            if not lip.is_pip:
                continue
            ix = min(int(lip.head_pos[0] / self.Lx * bins), bins - 1)
            iy = min(int(lip.head_pos[1] / self.Ly * bins), bins - 1)
            H[ix, iy] += (
                (1 + lip.lipid_type.pip_order * 0.5) * lip.lipid_type.mass
            )
        return gaussian_filter(H / ab, sigma=sigma)

    def _order_parameter_map(self, bins=70, sigma=1.5):
        """
        Mapa 2D de S_CH medio por celda [7, 8].
        Lo (gel): S~0.85-0.95 | Ld (fluido): S~0.60-0.75 en este modelo CG.
        """
        S_sum = np.zeros((bins, bins))
        C = np.zeros((bins, bins))
        for lip in self.outer_leaflet + self.inner_leaflet:
            ix = min(int(lip.head_pos[0] / self.Lx * bins), bins - 1)
            iy = min(int(lip.head_pos[1] / self.Ly * bins), bins - 1)
            S_sum[ix, iy] += lip.order_param
            C[ix, iy] += 1
        with np.errstate(all="ignore"):
            return gaussian_filter(np.where(C > 0, S_sum / C, 0), sigma=sigma)

    def _midplane_map(self, bins=70):
        """
        Plano medio local z_mid(x,y) = (z_sup + z_inf) / 2.
        Suavizado gaussiano para reducir ruido de bins con pocos lipidos.
        Necesario para corregir el efecto de la curvatura Helfrich [4].
        """
        S_s, C_s = np.zeros((bins, bins)), np.zeros((bins, bins))
        S_i, C_i = np.zeros((bins, bins)), np.zeros((bins, bins))
        for lip in self.outer_leaflet:
            ix = min(int(lip.head_pos[0] / self.Lx * bins), bins - 1)
            iy = min(int(lip.head_pos[1] / self.Ly * bins), bins - 1)
            S_s[ix, iy] += lip.head_pos[2]
            C_s[ix, iy] += 1
        for lip in self.inner_leaflet:
            ix = min(int(lip.head_pos[0] / self.Lx * bins), bins - 1)
            iy = min(int(lip.head_pos[1] / self.Ly * bins), bins - 1)
            S_i[ix, iy] += lip.head_pos[2]
            C_i[ix, iy] += 1
        S_s_sm = gaussian_filter(S_s, sigma=2.5)
        C_s_sm = gaussian_filter(C_s, sigma=2.5)
        S_i_sm = gaussian_filter(S_i, sigma=2.5)
        C_i_sm = gaussian_filter(C_i, sigma=2.5)
        with np.errstate(all="ignore"):
            z_s = np.where(C_s_sm > 0.1, S_s_sm / C_s_sm, self.geometry.z_outer)
            z_i = np.where(C_i_sm > 0.1, S_i_sm / C_i_sm, self.geometry.z_inner)
        return (z_s + z_i) / 2

    def _interdigitation_map(self, bins=70, sigma=2.0):
        """
        Penetracion normalizada de colas acil trans-leaflet [9].

        Metrica continua: score = max(0, penetracion) / longitud_cadena_local
        Donde penetracion = distancia del carbon terminal mas alla del plano
        medio LOCAL (corregido por curvatura Helfrich).

        Lo (gel, SM C24): score ~0.45-0.60 (cadenas largas)
        Ld (fluido): score ~0.25-0.40

        Referencia: Chaisson, Heberle, Doktorova, J.Chem.Inf.Model. 2025 [9]
        """
        z_mid = self._midplane_map(bins=bins)
        score_sum = np.zeros((bins, bins))
        count = np.zeros((bins, bins))

        for lip in self.outer_leaflet + self.inner_leaflet:
            ix = min(int(lip.head_pos[0] / self.Lx * bins), bins - 1)
            iy = min(int(lip.head_pos[1] / self.Ly * bins), bins - 1)
            z_ref = z_mid[ix, iy]
            z_glyc = lip.glycerol_pos[2]

            for tail in [lip.tail1, lip.tail2]:
                if not tail or len(tail) < 2:
                    continue
                z_end = tail[-1][2]
                chain_half = abs(z_glyc - z_ref)
                if chain_half < 1.0:
                    continue
                pen = (z_ref - z_end) if lip.leaflet == "sup" else (z_end - z_ref)
                score = max(0.0, pen / chain_half)
                score_sum[ix, iy] += score
                count[ix, iy] += 1

        with np.errstate(all="ignore"):
            raw = np.where(count > 0, score_sum / count, 0)
        return gaussian_filter(raw, sigma=sigma)

    def _z_profile(self, bins=160, sigma=0.8):
        todos = self.outer_leaflet + self.inner_leaflet
        zs = np.array([l.head_pos[2] for l in todos])
        ms = np.array([l.lipid_type.mass for l in todos])
        edges = np.linspace(zs.min() - 3, zs.max() + 3, bins + 1)
        H = np.zeros(bins)
        for z, m in zip(zs, ms):
            ix = np.clip(np.searchsorted(edges, z) - 1, 0, bins - 1)
            H[ix] += m
        return 0.5 * (edges[:-1] + edges[1:]), gaussian_filter(H, sigma=sigma)

    def _xz_projection(self, bx=220, bz=110, sigma=1.0):
        todos = self.outer_leaflet + self.inner_leaflet
        xs = np.array([l.head_pos[0] for l in todos]) / 10
        zs = np.array([l.head_pos[2] for l in todos]) / 10
        g = self.geometry
        ms = np.array([
            l.lipid_type.mass * (
                1.5 if abs(l.head_pos[2] - (
                    g.z_outer if l.leaflet == "sup" else g.z_inner
                )) < 5 else 0.5
            )
            for l in todos
        ])
        zr = [zs.min() - 0.3, zs.max() + 0.3]
        H, xe, ze = np.histogram2d(
            xs, zs, bins=[bx, bz],
            range=[[0, self.Lx / 10], zr], weights=ms,
        )
        return gaussian_filter(H, sigma=sigma), xe, ze

    def _volumetric_density(self, bins_xy=55, bins_z=40):
        """
        Densidad de masa 3D con coordenadas Z relativas al plano medio local.
        Elimina el efecto de la ondulacion Helfrich en los slices [4].
        """
        todos = self.outer_leaflet + self.inner_leaflet
        xs = np.array([l.head_pos[0] for l in todos]) / 10
        ys = np.array([l.head_pos[1] for l in todos]) / 10
        ms = np.array([l.lipid_type.mass for l in todos])

        z_mid_grid = self._midplane_map(bins=bins_xy)
        zs_rel = np.zeros(len(todos))
        for i, l in enumerate(todos):
            ix = min(int(l.head_pos[0] / self.Lx * bins_xy), bins_xy - 1)
            iy = min(int(l.head_pos[1] / self.Ly * bins_xy), bins_xy - 1)
            zs_rel[i] = (l.head_pos[2] - z_mid_grid[ix, iy]) / 10

        g = self.geometry
        z_half = (g.total_thick / 10) / 2 + 0.5
        H, edges = np.histogramdd(
            np.column_stack([xs, ys, zs_rel]),
            bins=[bins_xy, bins_xy, bins_z],
            range=[[0, self.Lx / 10], [0, self.Ly / 10], [-z_half, z_half]],
            weights=ms,
        )
        return gaussian_filter(H, sigma=[1.2, 1.2, 0.8]), edges

    @staticmethod
    def _add_colorbar(fig, im, ax, label, fmt="%.1f"):
        cbar = fig.colorbar(im, ax=ax, shrink=0.88, pad=0.02, format=fmt)
        cbar.set_label(label, fontsize=8)
        cbar.ax.tick_params(labelsize=7)
        return cbar

    @staticmethod
    def _decorate(ax, title, subtitle="", xlabel="X (nm)", ylabel="Y (nm)"):
        ax.set_title(
            title + (("\n%s" % subtitle) if subtitle else ""),
            fontsize=9, fontweight="bold", pad=5,
        )
        ax.set_xlabel(xlabel, fontsize=8)
        ax.set_ylabel(ylabel, fontsize=8)
        ax.xaxis.set_minor_locator(AutoMinorLocator())
        ax.yaxis.set_minor_locator(AutoMinorLocator())

    def _draw_rafts(self, ax, rafts, color="#e07b00"):
        for raft in rafts:
            cx = np.mean([l.head_pos[0] for l in raft]) / 10
            cy = np.mean([l.head_pos[1] for l in raft]) / 10
            r = (
                np.std([l.head_pos[0] for l in raft])
                + np.std([l.head_pos[1] for l in raft])
            ) / 20 + 0.8
            ax.add_patch(plt.Circle(
                (cx, cy), r, color=color,
                fill=False, lw=1.4, ls="--", alpha=0.85,
            ))
            ax.text(cx, cy, "⬡", fontsize=8, ha="center",
                    va="center", color=color, alpha=0.8)

    def _save_figure(self, fig, filename):
        out_dir = os.path.join(OUTPUT_DIR, "seed%04d" % self.seed)
        os.makedirs(out_dir, exist_ok=True)
        path = os.path.join(out_dir, filename)
        fig.savefig(path, dpi=FIG_DPI, bbox_inches="tight", facecolor="white")
        plt.close(fig)
        print("  -> %s" % filename)

    def plot_geometry_detailed(self):
        """
        Vista lateral coherente de la bicapa en una sola figura.
        Colas clipeadas en el plano medio → las dos monocapas
        se leen limpiamente sin entrelazamiento.
        Alpha proporcional a profundidad-Y para efecto de perspectiva.
        """
        g = self.geometry
        Lx_nm = self.Lx / 10
        x_center = Lx_nm / 2
        width_nm = 15.0
        y_depth_nm = 2.0
        x_start = x_center - width_nm / 2
        x_end = x_center + width_nm / 2
        mid_y = (self.Ly / 10) / 2

        def in_slice(lip):
            return (
                x_start * 10 <= lip.head_pos[0] <= x_end * 10
                and abs(lip.head_pos[1] / 10 - mid_y) < y_depth_nm / 2
            )

        lipids_slice = [
            l for l in self.outer_leaflet + self.inner_leaflet if in_slice(l)
        ]

        with plt.rc_context(PLT_STYLE):
            fig, ax = plt.subplots(figsize=(18, 11))
            fig.patch.set_facecolor("white")

            fig.suptitle(
                "Estructura Molecular de la Bicapa — Vista Lateral (seed=%d)\n"
                "Franja: %.1f-%.1f nm | Grosor: %.0f A | kc=%.0f kBT nm2 | sigma=%.3f kBT/nm2"
                % (self.seed, x_start, x_end,
                   g.total_thick, self.bending_modulus, self.surface_tension),
                fontsize=13, fontweight="bold", y=0.98,
            )

            # Fondos por zona
            ax.axhspan(g.z_inner / 10, g.z_outer / 10,
                       color="#f5f5dc", alpha=0.5, zorder=0)
            ax.axhspan(g.z_outer / 10, g.z_outer / 10 + 1.5,
                       color="#e8f5e8", alpha=0.6, zorder=0)
            ax.axhspan(g.z_inner / 10 - 1.5, g.z_inner / 10,
                       color="#f5e8e8", alpha=0.6, zorder=0)

            # Lineas de referencia
            ax.axhline(g.z_outer / 10, color="#2dc653", lw=2.0,
                       ls="-", alpha=0.9, label="Cabezas externas")
            ax.axhline(g.z_inner / 10, color="#e63946", lw=2.0,
                       ls="-", alpha=0.9, label="Cabezas internas")
            ax.axhline(0, color="#666666", lw=1.5,
                       ls="--", alpha=0.7, label="Plano medio")

            y_range = max(y_depth_nm * 10, 1.0)

            def alpha_depth(y_pos):
                return 0.45 + 0.50 * abs(y_pos - mid_y * 10) / y_range

            # Renderizar inf primero, sup encima (layering correcto)
            lipids_sorted = sorted(lipids_slice, key=lambda l: l.head_pos[1])

            for lip in lipids_sorted:
                alpha = alpha_depth(lip.head_pos[1])
                ltype = lip.lipid_type

                # Enlace cabeza-glicerol
                ax.plot(
                    [lip.head_pos[0] / 10, lip.glycerol_pos[0] / 10],
                    [lip.head_pos[2] / 10, lip.glycerol_pos[2] / 10],
                    color="#333333", lw=3.5, alpha=alpha, zorder=4,
                    solid_capstyle="round",
                )

                # Cabeza: circulo proporcional a hg_radius [17]
                ax.scatter(
                    lip.head_pos[0] / 10, lip.head_pos[2] / 10,
                    s=(ltype.hg_radius * 4.5) ** 2,
                    c=ltype.color, alpha=alpha, zorder=5,
                    edgecolors="white", linewidths=1.0,
                )

                # Glicerol
                ax.scatter(
                    lip.glycerol_pos[0] / 10, lip.glycerol_pos[2] / 10,
                    s=30, c="#1a1a1a", marker="s", alpha=alpha, zorder=4,
                    edgecolors="white", linewidths=0.8,
                )

                # Limites de clipeo: cola queda en su hemileaflet
                if lip.leaflet == "sup":
                    z_lo, z_hi = -0.3, g.z_outer / 10 + 0.5
                else:
                    z_lo, z_hi = g.z_inner / 10 - 0.5, 0.3

                # Cola sn1
                if lip.tail1:
                    pts = np.array(lip.tail1) / 10
                    mask = (pts[:, 2] >= z_lo) & (pts[:, 2] <= z_hi)
                    if np.any(mask):
                        ax.plot(
                            pts[mask, 0], pts[mask, 2],
                            color=ltype.color_tail1, lw=2.5,
                            alpha=alpha, zorder=3, solid_capstyle="round",
                        )
                    # Kink de doble enlace [2]
                    if ltype.ndb[0] > 0 and ltype.dbpos[0]:
                        ki = int(ltype.dbpos[0] / ltype.nc[0] * len(lip.tail1))
                        if 0 <= ki < len(lip.tail1):
                            zk = lip.tail1[ki][2] / 10
                            if z_lo <= zk <= z_hi:
                                ax.scatter(
                                    lip.tail1[ki][0] / 10, zk,
                                    s=50, c="#cc2200", zorder=6,
                                    alpha=alpha, edgecolors="white",
                                    linewidths=1.0, marker="o",
                                )

                # Cola sn2 (mas tenue)
                if lip.tail2:
                    pts = np.array(lip.tail2) / 10
                    mask = (pts[:, 2] >= z_lo) & (pts[:, 2] <= z_hi)
                    if np.any(mask):
                        ax.plot(
                            pts[mask, 0], pts[mask, 2],
                            color=ltype.color_tail2, lw=1.8,
                            alpha=alpha * 0.7, zorder=2,
                            solid_capstyle="round",
                        )

            # Leyenda de tipos lipidicos
            tipos_presentes = {lip.lipid_type for lip in lipids_slice}
            if tipos_presentes:
                handles = [
                    mpatches.Patch(
                        facecolor=t.color, edgecolor="black",
                        label=t.name, alpha=0.9,
                    )
                    for t in sorted(tipos_presentes, key=lambda x: x.name)
                ]
                leg1 = ax.legend(
                    handles=handles, loc="upper right", fontsize=9,
                    framealpha=0.95, title="Tipos de lipidos",
                    title_fontsize=10,
                )
                ax.add_artist(leg1)

            # Leyenda de elementos geometricos
            geom_elements = [
                mpatches.Patch(facecolor="#f5f5dc", edgecolor="gray",
                               label="Nucleo hidrofobico"),
                plt.Line2D([0], [0], color="#333333", lw=3,
                           label="Enlace cabeza-glicerol"),
                plt.Line2D([0], [0], color="gray", lw=2,
                           label="Cola acil"),
                mpatches.Patch(facecolor="#cc2200", edgecolor="white",
                               label="Doble enlace (kink)"),
            ]
            ax.legend(handles=geom_elements, loc="lower right",
                      fontsize=9, framealpha=0.95)

            # Flecha de grosor
            mid_z = (g.z_outer + g.z_inner) / 20
            ax.annotate(
                "", xy=(x_start + 0.5, g.z_outer / 10),
                xytext=(x_start + 0.5, g.z_inner / 10),
                arrowprops=dict(arrowstyle="<->", color="black", lw=2.5),
            )
            ax.text(
                x_start + 0.85, mid_z, "%.0f A" % g.total_thick,
                fontsize=12, va="center", fontweight="bold",
                bbox=dict(boxstyle="round,pad=0.4", facecolor="white",
                          edgecolor="black", alpha=0.95),
            )

            # Anotaciones de zona
            ax.text(
                0.02, 0.98,
                "Monocapa externa (sup)\n"
                "+ Cabezas polares\n"
                "+ Glicerol (cuadrado)\n"
                "+ Colas extendidas",
                transform=ax.transAxes, fontsize=9, va="top", ha="left",
                bbox=dict(boxstyle="round,pad=0.5", facecolor="#e8f5e8",
                          edgecolor="#2dc653", alpha=0.9),
            )
            ax.text(
                0.02, 0.02,
                "Monocapa interna (inf)\n"
                "+ Cabezas polares\n"
                "+ Glicerol (cuadrado)\n"
                "+ Colas extendidas",
                transform=ax.transAxes, fontsize=9, va="bottom", ha="left",
                bbox=dict(boxstyle="round,pad=0.5", facecolor="#f5e8e8",
                          edgecolor="#e63946", alpha=0.9),
            )

            ax.set_xlabel("X (nm)", fontsize=12)
            ax.set_ylabel("Z (nm)", fontsize=12)
            ax.set_xlim(x_start, x_end)
            ax.set_ylim(g.z_inner / 10 - 2.0, g.z_outer / 10 + 2.0)
            ax.grid(True, alpha=0.3)

            plt.tight_layout()
            self._save_figure(fig, "fig5_geometria.png")

    def plot_cryoET(self):
        ext = [0, self.Lx / 10, 0, self.Ly / 10]
        g = self.geometry
        with plt.rc_context(PLT_STYLE):
            fig, axes = plt.subplots(
                1, 2, figsize=(18, 8),
                gridspec_kw={"width_ratios": [1, 1.6]},
            )
            fig.suptitle(
                "Bicapa lipidica — Membrana plasmatica (seed=%d)\n"
                "%.0fx%.0f nm | Grosor %.1f A | kc=%.0f kBT nm2"
                % (
                    self.seed,
                    self.Lx / 10, self.Ly / 10,
                    g.total_thick, self.bending_modulus,
                ),
                fontsize=11, fontweight="bold",
            )

            ax = axes[0]
            D = (self._density_map(self.outer_leaflet)
                 + self._density_map(self.inner_leaflet))
            im = ax.imshow(D.T, origin="lower", extent=ext,
                           cmap="gray", aspect="equal")
            self._add_colorbar(fig, im, ax, "Da/A2")
            self._decorate(ax, "Proyeccion XY", "(simula cryo-ET)")

            ax2 = axes[1]
            Hxz, xe, ze = self._xz_projection(bx=280, bz=140)
            im2 = ax2.imshow(
                Hxz.T, origin="lower",
                extent=[xe[0], xe[-1], ze[0], ze[-1]],
                cmap="gray", aspect="auto",
            )
            self._add_colorbar(fig, im2, ax2, "Densidad (u.a.)")
            mid_z = (g.z_outer + g.z_inner) / 20
            for zl, col, lbl in [
                (g.z_outer / 10, "#2dc653", "Cabezas sup"),
                (g.z_inner / 10, "#e63946", "Cabezas inf"),
                (0.0, "#3a86ff", "Plano medio"),
            ]:
                ax2.axhline(zl, color=col, lw=1.0, ls="--",
                            alpha=0.85, label=lbl)
            x0a = xe[-1] * 0.03
            ax2.annotate(
                "", xy=(x0a, g.z_outer / 10), xytext=(x0a, g.z_inner / 10),
                arrowprops=dict(arrowstyle="<->", color="black", lw=1.2),
            )
            ax2.text(
                x0a + xe[-1] * 0.015, mid_z, "%.0f A" % g.total_thick,
                fontsize=9, va="center", fontweight="bold",
            )
            ax2.legend(fontsize=8, loc="upper right")
            self._decorate(ax2, "Seccion XZ",
                           "Banda densa sup + nucleo claro + banda densa inf")

            fig.tight_layout()
            self._save_figure(fig, "fig1_cryoET.png")

    def plot_domains(self):
        ext = [0, self.Lx / 10, 0, self.Ly / 10]
        g = self.geometry
        with plt.rc_context(PLT_STYLE):
            fig, axes = plt.subplots(2, 2, figsize=(14, 12))
            fig.suptitle(
                "Dominios lipidicos (seed=%d)" % self.seed,
                fontsize=12, fontweight="bold",
            )

            for ax, lips, rafts, titulo, sub in [
                (axes[0, 0], self.outer_leaflet, self.rafts_outer,
                 "Balsas externas", "SM/CHOL/GM1"),
                (axes[0, 1], self.inner_leaflet, self.rafts_inner,
                 "Balsas internas", "PS/PI enriquecido"),
            ]:
                im = ax.imshow(
                    self._raft_fraction_map(lips).T, origin="lower",
                    extent=ext, cmap=CMAP_RAFT, aspect="equal", vmin=0, vmax=1,
                )
                self._add_colorbar(fig, im, ax, "Fraccion raft")
                self._draw_rafts(ax, rafts)
                self._decorate(ax, titulo, sub)

            ax3 = axes[1, 0]
            G = self._thickness_map()
            im3 = ax3.imshow(G.T, origin="lower", extent=ext,
                             cmap=CMAP_THICK, aspect="equal")
            self._add_colorbar(fig, im3, ax3, "Grosor (A)")
            self._draw_rafts(ax3, self.rafts_outer, color="#888888")
            ax3.text(
                0.02, 0.04,
                "mu=%.1f A | hidro=%.1f A" % (G.mean(), g.hydro_thick),
                transform=ax3.transAxes, fontsize=7.5, va="bottom",
                bbox=dict(boxstyle="round,pad=0.3",
                          facecolor="white", edgecolor="#cccccc", alpha=0.9),
            )
            self._decorate(ax3, "Grosor local", "Zonas raft = mas gruesas")

            ax4 = axes[1, 1]
            R = self._roughness_map(self.outer_leaflet)
            im4 = ax4.imshow(R.T, origin="lower", extent=ext,
                             cmap="YlOrRd", aspect="equal")
            self._add_colorbar(fig, im4, ax4, "sigma_z (A)")
            ax4.text(
                0.02, 0.96, "Fluido ~1.8 A | Gel ~0.6 A",
                transform=ax4.transAxes, fontsize=7.5, va="top",
                bbox=dict(boxstyle="round,pad=0.3",
                          facecolor="white", edgecolor="#cccccc", alpha=0.9),
            )
            self._decorate(ax4, "Rugosidad externa", "Fluctuaciones termicas")

            fig.tight_layout()
            self._save_figure(fig, "fig2_dominios.png")

    def plot_pips(self):
        pip_types = ["PI", "PI3P", "PI4P", "PI5P", "PI34P2", "PIP2", "PIP3"]
        cnt_pip = {
            t: sum(1 for l in self.inner_leaflet
                   if l.lipid_type == LIPID_TYPES[t])
            for t in pip_types
        }
        cnt_pip = {k: v for k, v in cnt_pip.items() if v > 0}
        if not cnt_pip:
            print("  Sin PIPs en monocapa interna")
            return

        ext = [0, self.Lx / 10, 0, self.Ly / 10]
        with plt.rc_context(PLT_STYLE):
            fig, axes = plt.subplots(1, 2, figsize=(16, 7))
            fig.suptitle(
                "Fosfoinositoles (seed=%d)" % self.seed,
                fontsize=12, fontweight="bold",
            )

            ax = axes[0]
            im = ax.imshow(
                self._pip_density_map().T, origin="lower", extent=ext,
                cmap="magma", aspect="equal",
            )
            self._add_colorbar(fig, im, ax, "Da/A2 (PIPs)")
            for pc in self.pip_clusters:
                cx = np.mean([l.head_pos[0] for l in pc]) / 10
                cy = np.mean([l.head_pos[1] for l in pc]) / 10
                r = (
                    np.std([l.head_pos[0] for l in pc])
                    + np.std([l.head_pos[1] for l in pc])
                ) / 20 + 0.6
                ax.add_patch(plt.Circle(
                    (cx, cy), r, color="black", fill=False, lw=1.5, ls=":"
                ))
            self._decorate(ax, "Densidad PIPs", "Circulos = clusters")

            ax2 = axes[1]
            tipos_ = sorted(cnt_pip, key=lambda t: LIPID_TYPES[t].pip_order)
            bars = ax2.bar(
                [LIPID_TYPES[t].name for t in tipos_],
                [cnt_pip[t] for t in tipos_],
                color=[LIPID_TYPES[t].color for t in tipos_],
                edgecolor="#333333", lw=0.8,
            )
            ax2.bar_label(bars, fontsize=9, padding=3)
            ax2.set_xlabel("Especie PIP")
            ax2.set_ylabel("N lipidos")
            ax2.tick_params(axis="x", rotation=35)
            ax2.set_title(
                "Recuento PIPs (monocapa interna)",
                fontsize=9, fontweight="bold",
            )

            fig.tight_layout()
            self._save_figure(fig, "fig3_pips.png")

    def plot_composition(self):
        g = self.geometry
        ext = [0, self.Lx / 10, 0, self.Ly / 10]
        with plt.rc_context(PLT_STYLE):
            fig = plt.figure(figsize=(20, 7))
            fig.suptitle(
                "Composicion y perfil Z (seed=%d)" % self.seed,
                fontsize=12, fontweight="bold",
            )
            gs = mgrid.GridSpec(
                1, 3, wspace=0.38, left=0.06, right=0.97,
                top=0.88, bottom=0.14,
            )

            ax = fig.add_subplot(gs[0])
            todos = self.outer_leaflet + self.inner_leaflet
            tipos = sorted({l.lipid_type for l in todos}, key=lambda x: x.name)
            n_s = len(self.outer_leaflet)
            n_i = len(self.inner_leaflet)
            cs = {
                lt: sum(1 for l in self.outer_leaflet
                        if l.lipid_type == lt) / n_s * 100
                for lt in tipos
            }
            ci = {
                lt: sum(1 for l in self.inner_leaflet
                        if l.lipid_type == lt) / n_i * 100
                for lt in tipos
            }
            x_pos = np.arange(len(tipos))
            w = 0.38
            ax.bar(x_pos - w / 2, [cs[lt] for lt in tipos], w,
                   color=[lt.color for lt in tipos],
                   alpha=0.95, edgecolor="#333333", lw=0.5, label="Superior")
            ax.bar(x_pos + w / 2, [ci[lt] for lt in tipos], w,
                   color=[lt.color for lt in tipos],
                   alpha=0.55, edgecolor="#333333", lw=0.5,
                   hatch="//", label="Inferior")
            ax.set_xticks(x_pos)
            ax.set_xticklabels([lt.name for lt in tipos],
                               rotation=40, ha="right")
            ax.set_ylabel("% en monocapa")
            ax.legend(fontsize=8)
            ax.set_title("Composicion lipidica\nSuperior vs Inferior",
                         fontsize=10, fontweight="bold")

            ax2 = fig.add_subplot(gs[1])
            zc, hz = self._z_profile()
            hn = hz / hz.max()
            ax2.fill_betweenx(zc / 10, 0, hn, alpha=0.40, color="#3a86ff")
            ax2.plot(hn, zc / 10, color="#1a5fbf", lw=1.8)
            for zl, col, lbl in [
                (g.z_outer / 10, "#2dc653",
                 "Cabezas sup (%.0f A)" % g.z_outer),
                (g.z_inner / 10, "#e63946",
                 "Cabezas inf (%.0f A)" % g.z_inner),
            ]:
                ax2.axhline(zl, color=col, lw=1.1, ls="--", label=lbl)
            ax2.axhspan(g.z_inner / 10, g.z_outer / 10,
                        color="#eeee88", alpha=0.3)
            ax2.legend(fontsize=7.5, loc="upper left")
            ax2.xaxis.set_minor_locator(AutoMinorLocator())
            ax2.yaxis.set_minor_locator(AutoMinorLocator())
            self._decorate(ax2, "Perfil densidad Z", "Patron pico-valle-pico",
                           "Densidad norm.", "Z (nm)")

            ax3 = fig.add_subplot(gs[2])
            Asim = (self._density_map(self.outer_leaflet, sigma=2.0)
                    - self._density_map(self.inner_leaflet, sigma=2.0))
            vabs = np.percentile(np.abs(Asim), 98)
            im3 = ax3.imshow(Asim.T, origin="lower", extent=ext,
                             cmap="RdBu_r", vmin=-vabs, vmax=vabs)
            self._add_colorbar(fig, im3, ax3, "rho_sup - rho_inf")
            self._decorate(ax3, "Asimetria sup - inf",
                           "Rojo = ext mayor | Azul = int mayor")

            self._save_figure(fig, "fig4_composicion.png")

    def plot_order_chains(self):
        """
        Figuras 6: S_CH [7,8] e interdigitacion [9].
        """
        ext = [0, self.Lx / 10, 0, self.Ly / 10]
        with plt.rc_context(PLT_STYLE):
            fig, axes = plt.subplots(1, 3, figsize=(20, 7))
            fig.suptitle(
                "Orden de cadenas acil e interdigitacion (seed=%d)\n"
                "Piggot et al. JCTC 2017 [7] . "
                "Chaisson, Heberle, Doktorova JCIM 2025 [9]"
                % self.seed,
                fontsize=11, fontweight="bold",
            )

            ax = axes[0]
            S = self._order_parameter_map()
            im = ax.imshow(S.T, origin="lower", extent=ext,
                           cmap=CMAP_ORDER, aspect="equal")
            self._add_colorbar(fig, im, ax, "S_CH (orden)")
            self._draw_rafts(ax, self.rafts_outer)
            ax.text(
                0.02, 0.04,
                "Lo (gel) > Ld (fluido)\nBalsas = mayor orden",
                transform=ax.transAxes, fontsize=7.5, va="bottom",
                bbox=dict(boxstyle="round,pad=0.3",
                          facecolor="white", edgecolor="#cccccc", alpha=0.9),
            )
            self._decorate(ax, "Parametro de orden S_CH",
                           "<(3cos2(theta)-1)/2> por celda")

            ax2 = axes[1]
            ID = self._interdigitation_map()
            vmax_id = max(ID.mean() + 2 * ID.std(), ID.max() * 0.5)
            im2 = ax2.imshow(ID.T, origin="lower", extent=ext,
                             cmap=CMAP_INTERDIG, aspect="equal",
                             vmin=0, vmax=vmax_id)
            self._add_colorbar(fig, im2, ax2, "Penetracion norm. (u.a.)", "%.2f")
            ax2.text(
                0.02, 0.04,
                "score = profundidad / longitud_cadena\nLo > Ld",
                transform=ax2.transAxes, fontsize=7.5, va="bottom",
                bbox=dict(boxstyle="round,pad=0.3",
                          facecolor="white", edgecolor="#cccccc", alpha=0.9),
            )
            self._decorate(ax2, "Interdigitacion trans-leaflet",
                           "Chaisson et al. 2025 [9]")

            ax3 = axes[2]
            todos = self.outer_leaflet + self.inner_leaflet
            s_gel   = [l.order_param for l in todos if l.lipid_type.phase == "gel"]
            s_fluid = [l.order_param for l in todos if l.lipid_type.phase == "fluid"]
            ax3.hist(s_fluid, bins=40, density=True, alpha=0.6,
                     label="Fluido (Ld)", color="#3a86ff",
                     histtype="stepfilled", ec="#1a5fbf")
            ax3.hist(s_gel, bins=40, density=True, alpha=0.6,
                     label="Gel (Lo)", color="#2dc653",
                     histtype="stepfilled", ec="#0a6e2d")
            ax3.axvline(np.mean(s_fluid) if s_fluid else 0,
                        color="#1a5fbf", lw=1.5, ls="--")
            ax3.axvline(np.mean(s_gel) if s_gel else 0,
                        color="#0a6e2d", lw=1.5, ls="--")
            ax3.set_xlabel("S_CH")
            ax3.set_ylabel("Densidad")
            ax3.legend(fontsize=8)
            ax3.set_title("Distribucion S_CH por fase\nGel vs Fluido",
                          fontsize=9, fontweight="bold")

            fig.tight_layout()
            self._save_figure(fig, "fig6_orden_cadenas.png")

    def plot_density_3d(self):
        """Slices ortogonales del volumen 3D de densidad [14, 15, 16]."""
        H, edges = self._volumetric_density(bins_xy=55, bins_z=40)
        ex, ey, ez = edges
        cx = 0.5 * (ex[:-1] + ex[1:])
        cy = 0.5 * (ey[:-1] + ey[1:])
        cz = 0.5 * (ez[:-1] + ez[1:])
        g = self.geometry

        with plt.rc_context(PLT_STYLE):
            fig, axes = plt.subplots(1, 3, figsize=(20, 6))
            fig.suptitle(
                "Densidad volumetrica 3D — slices ortogonales (seed=%d)\n"
                "Bicapa: banda densa (cabezas) + nucleo claro (colas) [14, 15]"
                % self.seed,
                fontsize=11, fontweight="bold",
            )

            ax = axes[0]
            Hxz = H[:, len(cy) // 2, :]
            im = ax.imshow(Hxz.T, origin="lower",
                           extent=[cx[0], cx[-1], cz[0], cz[-1]],
                           cmap="gray_r", aspect="auto")
            self._add_colorbar(fig, im, ax, "Densidad")
            for zl, col in [
                (g.z_outer / 10, "#2dc653"),
                (g.z_inner / 10, "#e63946"),
                (0.0, "#3a86ff"),
            ]:
                ax.axhline(zl, color=col, lw=0.9, ls="--", alpha=0.7)
            self._decorate(ax, "Slice XZ (Y=centro)",
                           "Z relativo al plano medio",
                           "X (nm)", "Z relativo (nm)")

            ax2 = axes[1]
            Hyz = H[len(cx) // 2, :, :]
            im2 = ax2.imshow(Hyz.T, origin="lower",
                             extent=[cy[0], cy[-1], cz[0], cz[-1]],
                             cmap="gray_r", aspect="auto")
            self._add_colorbar(fig, im2, ax2, "Densidad")
            for zl, col in [
                (g.z_outer / 10, "#2dc653"),
                (g.z_inner / 10, "#e63946"),
                (0.0, "#3a86ff"),
            ]:
                ax2.axhline(zl, color=col, lw=0.9, ls="--", alpha=0.7)
            self._decorate(ax2, "Slice YZ (X=centro)",
                           "Segunda vista transversal",
                           "Y (nm)", "Z relativo (nm)")

            ax3 = axes[2]
            z_lo = g.z_outer / 10 * 0.5
            z_hi = g.z_outer / 10 * 1.3
            iz_lo = max(0, np.searchsorted(cz, z_lo) - 1)
            iz_hi = min(len(cz) - 1, np.searchsorted(cz, z_hi))
            Hxy = H[:, :, iz_lo:iz_hi + 1].max(axis=2)
            im3 = ax3.imshow(Hxy.T, origin="lower",
                             extent=[cx[0], cx[-1], cy[0], cy[-1]],
                             cmap="gray_r", aspect="equal")
            self._add_colorbar(fig, im3, ax3, "Densidad (MIP)")
            self._draw_rafts(ax3, self.rafts_outer)
            self._decorate(
                ax3, "Proyeccion maxima XY (cabezas externas)",
                "MIP z=[%.1f, %.1f] nm" % (z_lo, z_hi),
            )

            fig.tight_layout()
            self._save_figure(fig, "fig7_densidad3d.png")

    def export_training(self, bins=64):
        """
        Exporta 11 canales numpy sin aumentar (augmentacion externa).
        El labels.json global acumula metadatos de todas las semillas.
        """
        d = os.path.join(OUTPUT_DIR, "training", "seed%04d" % self.seed)
        os.makedirs(d, exist_ok=True)

        channels = {
            "ch0_cryoET": (
                self._density_map(self.outer_leaflet, bins=bins)
                + self._density_map(self.inner_leaflet, bins=bins)
            ),
            "ch1_thickness":   self._thickness_map(bins=bins),
            "ch2_rough_outer": self._roughness_map(self.outer_leaflet, bins=bins),
            "ch3_rough_inner": self._roughness_map(self.inner_leaflet, bins=bins),
            "ch4_raft_outer":  self._raft_fraction_map(self.outer_leaflet, bins=bins),
            "ch5_raft_inner":  self._raft_fraction_map(self.inner_leaflet, bins=bins),
            "ch6_pip_density": self._pip_density_map(bins=bins),
            "ch8_asymmetry":   (
                self._density_map(self.outer_leaflet, bins=bins, sigma=2.0)
                - self._density_map(self.inner_leaflet, bins=bins, sigma=2.0)
            ),
            "ch10_order":      self._order_parameter_map(bins=bins),
            "ch11_interdig":   self._interdigitation_map(bins=bins),
        }
        Hxz, _, _ = self._xz_projection(bx=bins * 2, bz=bins)
        channels["ch9_xz_slice"] = Hxz

        for name, arr in channels.items():
            np.save(os.path.join(d, "%s.npy" % name), arr)

        g = self.geometry
        todos = self.outer_leaflet + self.inner_leaflet
        meta = {
            "seed": self.seed,
            "size_nm": [self.Lx / 10, self.Ly / 10],
            "kc_kBT_nm2": round(self.bending_modulus, 2),
            "sigma_kBT_nm2": round(self.surface_tension, 4),
            "grosor_total_A": round(g.total_thick, 2),
            "grosor_hidro_A": round(g.hydro_thick, 2),
            "n_sup": len(self.outer_leaflet),
            "n_inf": len(self.inner_leaflet),
            "n_balsas_s": len(self.rafts_outer),
            "n_balsas_i": len(self.rafts_inner),
            "n_pip_clusters": len(self.pip_clusters),
            "n_perturbadores": len(self.perturbations),
            "densidad_perturbadores": round(self.perturbation_density, 4),
            "S_CH_medio_gel": round(float(np.mean(
                [l.order_param for l in todos
                 if l.lipid_type.phase == "gel"]
            ) if any(l.lipid_type.phase == "gel" for l in todos) else 0), 3),
            "S_CH_medio_fluido": round(float(np.mean(
                [l.order_param for l in todos
                 if l.lipid_type.phase == "fluid"]
            ) if any(l.lipid_type.phase == "fluid" for l in todos) else 0), 3),
            "comp_outer": {t: round(f, 4) for t, f in self.comp_outer.items()},
            "comp_inner": {t: round(f, 4) for t, f in self.comp_inner.items()},
            "canales": {
                "ch0": "cryoET_imagen_densidad",
                "ch1": "grosor_local_bicapa",
                "ch2": "rugosidad_monocapa_externa",
                "ch3": "rugosidad_monocapa_interna",
                "ch4": "fraccion_raft_externa",
                "ch5": "fraccion_raft_interna",
                "ch6": "densidad_pip_total",
                "ch8": "asimetria_composicional",
                "ch9": "slice_xz_cryoET",
                "ch10": "parametro_orden_S_CH [7,8]",
                "ch11": "interdigitacion_trans_leaflet [9]",
            },
            "references": [
                "Smith et al. LiveCoMS 2019 [2]",
                "Helfrich 1973 [4]",
                "Pinigin Membranes 2022 [5]",
                "Chakraborty et al. PNAS 2020 [6]",
                "Piggot et al. JCTC 2017 [7]",
                "Chaisson et al. JCIM 2025 [9]",
            ],
        }

        labels_file = os.path.join(OUTPUT_DIR, "training", "labels.json")
        all_meta = []
        if os.path.exists(labels_file):
            with open(labels_file) as f:
                all_meta = json.load(f)
        all_meta = [m for m in all_meta if m.get("seed") != self.seed]
        all_meta.append(meta)
        with open(labels_file, "w") as f:
            json.dump(all_meta, f, indent=2)

        print("  -> training/seed%04d/ (%d canales)" % (self.seed, len(channels)))
        return d

    def save_all(self):
        print("  Figuras para seed=%d..." % self.seed)
        self.plot_cryoET()
        self.plot_domains()
        self.plot_pips()
        self.plot_composition()
        self.plot_geometry_detailed()
        self.plot_order_chains()
        self.plot_density_3d()

    @staticmethod
    def generate_dataset(seeds, size_nm=(50, 50), solo_training=False):
        """
        Genera dataset con lista de semillas.
        Cada semilla produce composicion, curvatura y dominios distintos.
        """
        seeds = list(seeds)
        print("Dataset: %d instantaneas de %dx%d nm" % (
            len(seeds), size_nm[0], size_nm[1]
        ))
        for seed in seeds:
            b = BicapaCryoET(size_nm=size_nm, seed=seed)
            b.build()
            b.export_training()
            if not solo_training:
                b.save_all()
        print("Listo en: %s/" % OUTPUT_DIR)


def main():
    """Genera dos muestras de ejemplo."""
    for seed in [27, 42]:
        b = BicapaCryoET(size_nm=(50, 50), seed=seed)
        b.build()
        b.save_all()
        b.export_training()
    print("\nListo en: %s/" % OUTPUT_DIR)


if __name__ == "__main__":
    main()

  seed=27 | 50x50 nm | lipidos: 12911 | rafts: 3/6 | pips: 2 | kc=28 kBT nm2 | sigma=0.007
  Figuras para seed=27...
  -> fig1_cryoET.png
  -> fig2_dominios.png
  -> fig3_pips.png
  -> fig4_composicion.png
  -> fig5_geometria.png
  -> fig6_orden_cadenas.png
  -> fig7_densidad3d.png
  -> training/seed0027/ (11 canales)
  seed=42 | 50x50 nm | lipidos: 12716 | rafts: 5/3 | pips: 2 | kc=27 kBT nm2 | sigma=0.009
  Figuras para seed=42...
  -> fig1_cryoET.png
  -> fig2_dominios.png
  -> fig3_pips.png
  -> fig4_composicion.png
  -> fig5_geometria.png
  -> fig6_orden_cadenas.png
  -> fig7_densidad3d.png
  -> training/seed0042/ (11 canales)

Listo en: CryoET/
